## Big function for drawing video

In [ ]:
import io
import math
from pathlib import Path

import duckdb
import imageio.v3 as iio
import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from PIL import Image as PILImage


def export_tiled_episode_videos(
    path: str | Path,
    camera_cols: list[str],
    state_cols: list[str] | None = None,
    action_cols: list[str] | None = None,
    output_dir: str | Path = "episode_videos",
    fps: int = 30,
    frame_stride: int = 10,
    n_frames: int | None = None,
    episode: int | None = None,
) -> None:
    """Export tiled camera MP4s, with an animated state/action plot below."""
    if not camera_cols:
        raise ValueError("camera_cols must contain at least one camera column")
    if frame_stride < 1:
        raise ValueError("frame_stride must be at least 1")

    path = Path(path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    state_cols = state_cols or []
    action_cols = action_cols or []

    def as_vector(value):
        if value is None:
            return None
        return np.asarray(value, dtype=np.float32).reshape(-1)

    def build_vector_series(rows, value_start_index, value_cols):
        series = []

        for col_index, col_name in enumerate(value_cols):
            vectors = [
                as_vector(row[value_start_index + col_index])
                for row in rows
            ]
            dimensions = max(
                (len(vector) for vector in vectors if vector is not None),
                default=0,
            )

            for dim in range(dimensions):
                values = np.full(len(rows), np.nan, dtype=np.float32)

                for frame_index, vector in enumerate(vectors):
                    if vector is not None and dim < len(vector):
                        values[frame_index] = vector[dim]

                label = col_name if dimensions == 1 else f"{col_name}[{dim}]"
                series.append((label, values))

        return series

    def get_y_limits(state_series, action_series):
        finite_parts = []

        for _, values in state_series + action_series:
            finite_values = values[np.isfinite(values)]
            if len(finite_values):
                finite_parts.append(finite_values)

        if not finite_parts:
            return 0.0, 1.0

        values = np.concatenate(finite_parts)
        padding = max((values.max() - values.min()) * 0.05, 1e-4)
        return values.min() - padding, values.max() + padding

    def make_plot_renderer(width, steps, state_series, action_series, episode_idx):
        if not state_series and not action_series:
            return None

        dpi = 100
        plot_height = 280

        figure = Figure(
            figsize=(width / dpi, plot_height / dpi),
            dpi=dpi,
            tight_layout=True,
        )
        canvas = FigureCanvasAgg(figure)
        ax = figure.add_subplot(1, 1, 1)

        y_limits = get_y_limits(state_series, action_series)

        def render(frame_index):
            ax.clear()

            # Solid state lines.
            for series_index, (label, values) in enumerate(state_series):
                ax.plot(
                    steps[: frame_index + 1],
                    values[: frame_index + 1],
                    color=f"C{series_index % 10}",
                    linewidth=1.4,
                    linestyle="-",
                    label=f"{label} — state",
                )

            # Dashed action lines. Reuse colours by component index.
            for series_index, (label, values) in enumerate(action_series):
                ax.plot(
                    steps[: frame_index + 1],
                    values[: frame_index + 1],
                    color=f"C{series_index % 10}",
                    linewidth=1.4,
                    linestyle="--",
                    label=f"{label} — action",
                )

            ax.axvline(
                steps[frame_index],
                color="black",
                linestyle=":",
                linewidth=1,
                alpha=0.7,
            )
            ax.set_xlim(steps[0], steps[-1])
            ax.set_ylim(*y_limits)
            ax.set_title(
                f"State and action — episode {episode_idx}, step {steps[frame_index]}",
                fontsize=9,
            )
            ax.set_xlabel("Source step", fontsize=8)
            ax.set_ylabel("Value", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.3)

            if len(state_series) + len(action_series) <= 16:
                ax.legend(fontsize=5, ncol=4, loc="upper right")

            canvas.draw()
            return np.asarray(canvas.buffer_rgba())[..., :3].copy()

        return render

    generated_videos = []
    skipped_existing = 0
    con = duckdb.connect()

    try:
        uuids = con.execute(
            """
            SELECT DISTINCT uuid
            FROM read_parquet(?)
            WHERE success
            ORDER BY uuid
            """,
            [str(path)],
        ).fetchnumpy()["uuid"]

        print(f"Found {len(uuids)} successful episodes")

        camera_selects = ",\n".join(
            f"{camera_col} AS camera_{index}"
            for index, camera_col in enumerate(camera_cols)
        )
        state_selects = ",\n".join(
            f"{state_col} AS state_{index}"
            for index, state_col in enumerate(state_cols)
        )
        action_selects = ",\n".join(
            f"{action_col} AS action_{index}"
            for index, action_col in enumerate(action_cols)
        )

        extra_selects = ",\n".join(
            select
            for select in (state_selects, action_selects)
            if select
        )
        extra_selects = f",\n{extra_selects}" if extra_selects else ""

        n_cameras = len(camera_cols)
        grid_cols = math.ceil(math.sqrt(n_cameras))
        grid_rows = math.ceil(n_cameras / grid_cols)
        limit_clause = f"LIMIT {n_frames}" if n_frames is not None else ""

        for episode_idx, uuid in enumerate(uuids):
            if episode is not None and episode_idx != episode:
                continue

            episode_dir = output_dir / str(uuid)
            video_path = episode_dir / "tiled.mp4"

            if video_path.exists():
                skipped_existing += 1
                continue

            episode_dir.mkdir(parents=True, exist_ok=True)

            rows = con.execute(
                f"""
                SELECT
                    step,
                    {camera_selects}
                    {extra_selects}
                FROM read_parquet(?)
                WHERE uuid = ?
                  AND step % ? = 0
                ORDER BY step
                {limit_clause}
                """,
                [str(path), str(uuid), frame_stride],
            ).fetchall()

            if not rows:
                print(f"Skipping episode {episode_idx}: no frames found")
                continue

            tile_width = 0
            tile_height = 0

            for row in rows:
                for image_data in row[1 : 1 + n_cameras]:
                    if image_data is None:
                        continue

                    with PILImage.open(io.BytesIO(image_data)) as image:
                        tile_width = max(tile_width, image.width)
                        tile_height = max(tile_height, image.height)

            if tile_width == 0 or tile_height == 0:
                print(f"Skipping episode {episode_idx}: all camera frames were empty")
                continue

            steps = np.asarray([row[0] for row in rows])

            state_start_index = 1 + n_cameras
            action_start_index = state_start_index + len(state_cols)

            state_series = build_vector_series(
                rows,
                state_start_index,
                state_cols,
            )
            action_series = build_vector_series(
                rows,
                action_start_index,
                action_cols,
            )

            tiled_width = grid_cols * tile_width
            render_plot = make_plot_renderer(
                tiled_width,
                steps,
                state_series,
                action_series,
                episode_idx,
            )

            frames = []

            for frame_index, row in enumerate(rows):
                tiled_frame = np.zeros(
                    (grid_rows * tile_height, tiled_width, 3),
                    dtype=np.uint8,
                )

                for camera_idx, image_data in enumerate(row[1 : 1 + n_cameras]):
                    if image_data is None:
                        continue

                    with PILImage.open(io.BytesIO(image_data)) as image:
                        image = image.convert("RGB")
                        image.thumbnail((tile_width, tile_height))
                        image_array = np.asarray(image)

                    tile_row = camera_idx // grid_cols
                    tile_col = camera_idx % grid_cols
                    y = tile_row * tile_height
                    x = tile_col * tile_width

                    y_offset = (tile_height - image_array.shape[0]) // 2
                    x_offset = (tile_width - image_array.shape[1]) // 2

                    tiled_frame[
                        y + y_offset : y + y_offset + image_array.shape[0],
                        x + x_offset : x + x_offset + image_array.shape[1],
                    ] = image_array

                if render_plot is not None:
                    plot_frame = render_plot(frame_index)
                    tiled_frame = np.vstack([tiled_frame, plot_frame])

                frames.append(tiled_frame)

            iio.imwrite(
                video_path,
                frames,
                fps=fps,
                codec="libx264",
                pixelformat="yuv420p",
            )

            generated_videos.append(video_path)

        print(f"\nGenerated {len(generated_videos)} new video(s):")
        for video_path in generated_videos:
            print(f"  {video_path}")

        print(f"\nSkipped {skipped_existing} existing video(s).")
        print("Done.")

    finally:
        con.close()

In [ ]:
import duckdb
# path = "/home/bien/Documents/Development/RCS/datasets/dataset_parquet/with_wood_cover/utn_fh_usbc_insertion"
path = "/home/bien/Documents/Development/RCS/robot-control-stack/examples/teleop/utn_fh_box_classification"
export_tiled_episode_videos(
    path=path,
    camera_cols=[
        "obs.frames.side.rgb.data",
        "obs.frames.wrist.rgb.data",
        # "obs.frames.digit_right_left.rgb.data",
        # "obs.frames.digit_right_left_blank.rgb.data",
        # "obs.frames.digit_right_right.rgb.data",
        # "obs.frames.digit_right_right_blank.rgb.data",
                
    ],
    state_cols=[
        "obs.right.joints",
        "obs.right.gripper",
    ],
    action_cols=[
        "info.right.absolute_action",
        "action.right.gripper",
    ],
    output_dir="episode_videos_with_state",
    frame_stride=5,
    episode=0
)

In [ ]:
path = "/home/bien/Documents/Development/RCS/robot-control-stack/examples/teleop/utn_fh_usbc_insertion"
duckdb.sql(
    "SELECT count(DISTINCT uuid) as successful "
    f"FROM read_parquet('{path}') "
    "WHERE success"
),duckdb.sql(
    "SELECT count(DISTINCT uuid) as n_episodes "
    f"FROM read_parquet('{path}') "
)

In [ ]:
import io
import duckdb
from PIL import Image as PILImage
from IPython.display import Image, display

uuids = duckdb.sql(
    f"SELECT DISTINCT uuid FROM read_parquet('{path}') ORDER BY uuid WHERE success"
).fetchnumpy()
episode = 0
n_frames = 1000
frame_stride = 5


uuid1 = uuids["uuid"][episode]
rel = duckdb.read_parquet(path)

count = rel.filter(f"uuid='{uuid1}'").count("*").fetchone()[0]
info = rel.filter(f"uuid='{uuid1}'").select("info").fetchone()[0]
success = rel.filter(f"uuid='{uuid1}'").select("success").fetchone()[0]

print(f"episode uuid: {uuid1}, success: {success}, n_steps: {count}, info: {info}")

steps = list(range(0, n_frames * frame_stride, frame_stride))

frames = (
    rel
    .filter(f"uuid='{uuid1}' and step in ({','.join(map(str, steps))})")
    .select("step, obs.frames.side.rgb.data AS img_data")
    .order("step")
    .fetchall()
)

print(f"Loaded {len(frames)} frames")

pil_frames = []

for step, img_data in frames:
    img = PILImage.open(io.BytesIO(img_data)).convert("RGB")
    pil_frames.append(img)

# Save to an in-memory GIF
gif_buffer = io.BytesIO()

pil_frames[0].save(
    gif_buffer,
    format="GIF",
    save_all=True,
    append_images=pil_frames[1:],
    duration=50,   # milliseconds per frame; 50 ms = 20 FPS
    loop=0,
)

gif_buffer.seek(0)

display(Image(data=gif_buffer.read(), format="gif"))

In [ ]:
import numpy as np
import pandas as pd
import duckdb
from matplotlib import pyplot as plt

episode = 13
uuids = duckdb.sql(
    f"SELECT DISTINCT uuid FROM read_parquet('{path}')"
).fetchnumpy()
episode = 0
n_frames = 400
frame_stride = 3
uuid1 = uuids["uuid"][episode]

df = duckdb.sql(f"""
    SELECT
        step,
        obs.right.tquat AS obs_tquat,
        action.right.tquat AS action_tquat,
        obs.right.gripper AS obs_gripper,
        action.right.gripper AS action_gripper,
        obs.right.joints AS obs_joints,
        info.right.absolute_action as absolute_action
    FROM read_parquet('{path}')
    WHERE uuid = '{uuid1}'
    ORDER BY step
""").df()

print(f"episode uuid: {uuid1}, n_steps: {len(df)}")

# Expand quaternion columns into x/y/z/w components
obs_tquat = np.stack(df["obs_tquat"][1:].to_numpy())
# print(df["action_tquat"].shape)
action_tquat = np.stack(df["action_tquat"][1:].to_numpy())
quat_names = ["x", "y", "z", "w"]
steps = df["step"][1:]

fig, ax = plt.subplots(1, 5, figsize=(26, 4), sharex=True)

for i, name in enumerate(quat_names):
    ax[i].plot(steps, obs_tquat[:, i], label="obs", linewidth=2)
    ax[i].plot(steps, action_tquat[:, i], label="action", linewidth=2, linestyle="--")
    ax[i].set_title(f"right.tquat.{name}")
    ax[i].set_xlabel("step")
    ax[i].grid(True, alpha=0.3)
    ax[i].legend()

ax[4].plot(steps, df["obs_gripper"][1:], label="obs", linewidth=2)
ax[4].plot(steps, df["action_gripper"][1:], label="action", linewidth=2, linestyle="--")
ax[4].set_title("right.gripper")
ax[4].set_xlabel("step")
ax[4].grid(True, alpha=0.3)
ax[4].legend()

plt.suptitle(f"UUID: {uuid1}", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Source-format equivalent:
# obs.right.joints  -> state
# info.right.absolute_action -> action
#
# Assumes `state`, `action`, `df`, `episode_id`, and `joint_labels` are defined.
joint_labels = {
    0: "joint_1",
    1: "joint_2",
    2: "joint_3",
    3: "joint_4",
    4: "joint_5",
    5: "joint_6",
    6: "joint_7",
}
state = np.stack(df["obs_joints"][1:].to_numpy())
action = np.stack(df["absolute_action"][1:].to_numpy())
num_dims = min(state.shape[1], action.shape[1])
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
axes = axes.ravel()

for dim in range(num_dims):
    joint_name = joint_labels.get(dim, f"joint_{dim + 1}")

    axes[dim].plot(
        steps,
        state[:, dim],
        label="obs",
        linewidth=2,
    )
    axes[dim].plot(
        steps,
        action[:, dim],
        label="absolute_action",
        linewidth=2,
        linestyle=":",  # dotted action line
    )

    axes[dim].set_title(joint_name)
    axes[dim].set_xlabel("frame index")
    axes[dim].grid(True, alpha=0.3)
    axes[dim].legend()

# Hide unused axes if the vectors have fewer than 8 dimensions
for ax in axes[num_dims:]:
    ax.set_visible(False)

plt.suptitle(f"Episode {episode}: observation joints vs absolute action", y=1.02)
plt.tight_layout()
plt.show()

# Save the <key> video from all episodes with stride as mp4

In [ ]:
export_tiled_episode_videos(
    path=path,
    camera_cols=[
        "obs.frames.side.rgb.data",
        "obs.frames.wrist.rgb.data",
        "obs.frames.digit_right_left.rgb.data",
        "obs.frames.digit_right_left_blank.rgb.data",
        "obs.frames.digit_right_right.rgb.data",
        "obs.frames.digit_right_right_blank.rgb.data",
    ],
)

# DANGER ZONE: DELETE EPISODE

In [ ]:
import duckdb
import shutil
from pathlib import Path
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

path = Path(
    "/home/bien/Documents/Development/RCS/robot-control-stack/"
    "examples/teleop/utn_fh_usbc_insertion"
)
# Removed from utn_fh_usbc_insertion


# Removed uuids from original utn_usbc_insertion dataset
# "ebebe18d4df249f7b59ff5ae18df81e1",
# "fbb2cd6390624d89a07a9c01817ebc3a",
# "7caf079e73d045eb93a38988522522e9",
# "74e83fbe08a24d56bd000ff5712e9237",
# "216ced430c43437db9864dc4709f446e",
# "264ef8ca2c684704964dff423a4790d2",
# "58754f76f81c47f8be52ed93c16596ea",
# "462514e9e5934ed6b72911e237aa9353",
# "6d7227dc64a94c029cd0b81ce61297f0",
# "d8110c285e09443ca239afda30b32adc"

# Removed from usbc_tape
# "d54a0a6da88d4a66b9f77c2c5c640177",
# "cefbb19ad61a459c929bffbf448f518d"

# Removed from utn_usb_tape_insertion
    # "1da6ecfa04a842e0b6e4c66b880d4005",
    # "7dc2bac9782248209130a13ec9180b66",
    # "21f6251a6e754841b0cebf7c43ce0b6d",
    # "65f8f1680bf446ccb6194592b1cc0bbc",
    # "427fd2c845094f8cb7aff3efd126f666",
    # "690f76bc171d431cbe3393624d015932",
    # "8323ed6195fc49e6bf76658de3ac64e3",
    # "ae39d39ca2034125b2e688830bbdd683",
    # "b2c9a1f4411b473c8eaa878fc62d467e",
    # "b4360c2184664a71b368708579450d04",
uuids_to_remove = [
    "0dbe2aaa673e4d919bf2e4bd67e32366",
    "1e98af40e19e4bfb9e1797f7ecd11498",
    "24cb679733014bf28e8a765d075b4b59",
    "485ef65630a540d4947754cfc3a66751",
    "b7d97629adb64ba1ae5377cc0136eb99",
    "d6e20f1011c54992a2310de2ee19ae50",
    "1591c892c38c48a5a07f8647d40be526",
    "abfad93371194d74b0d7a3ce386c5101",
    "e2fa33c2a4554ed8a71908f53ef28faa",
    "fc0ed28249d046f59ca1cdde01e1bca2",
]


if not uuids_to_remove:
    raise ValueError("uuids_to_remove cannot be empty.")

if not path.is_dir():
    raise ValueError("This version expects `path` to be a dataset directory.")

temp_path = path.with_name(path.name + "_filtered_tmp")
backup_path = path.with_name(path.name + "_backup")
validation_path = path.with_name(path.name + "_consolidation_validation_tmp")

if temp_path.exists():
    shutil.rmtree(temp_path)

if backup_path.exists():
    raise FileExistsError(f"Backup already exists: {backup_path}")

if validation_path.exists():
    shutil.rmtree(validation_path)

source_files = sorted(path.rglob("*.parquet"))
if not source_files:
    raise FileNotFoundError(f"No Parquet files found in {path}")

bad_uuid_values = pa.array(uuids_to_remove, type=pa.string())


def count_rows_and_matches(files: list[Path]) -> tuple[int, int]:
    """Count total rows and rows whose UUID is scheduled for removal."""
    total_rows = 0
    matching_rows = 0

    for parquet_file in files:
        parquet = pq.ParquetFile(parquet_file)

        if "uuid" not in parquet.schema_arrow.names:
            raise ValueError(f"Missing 'uuid' column: {parquet_file}")

        for batch in parquet.iter_batches(columns=["uuid"]):
            uuid_column = batch.column(0)
            matches = pc.is_in(uuid_column, value_set=bad_uuid_values)

            total_rows += batch.num_rows
            matching_rows += pc.sum(matches).as_py() or 0

    return total_rows, matching_rows


def filter_parquet_file(source_file: Path, destination_file: Path) -> int:
    """Filter one Parquet fragment while preserving its Arrow schema."""
    source_parquet = pq.ParquetFile(source_file)
    source_schema = source_parquet.schema_arrow
    uuid_index = source_schema.get_field_index("uuid")

    if uuid_index == -1:
        raise ValueError(f"Missing 'uuid' column: {source_file}")

    destination_file.parent.mkdir(parents=True, exist_ok=True)
    removed_rows = 0

    # The schema is taken directly from the original fragment, rather than
    # inferred through DuckDB. This preserves nested observation/image fields.
    with pq.ParquetWriter(
        destination_file,
        source_schema,
        compression="zstd",
    ) as writer:
        # Do not use ``iter_batches`` here.  It asks Parquet to produce
        # RecordBatches directly, which PyArrow cannot do for some nested
        # schemas (``ArrowNotImplementedError: Nested data conversions not
        # implemented for chunked array outputs``).  Reading one row group
        # produces an Arrow Table instead, whose ChunkedArrays support those
        # nested fields.
        for row_group_index in range(source_parquet.num_row_groups):
            row_group = source_parquet.read_row_group(row_group_index)
            uuid_column = row_group.column(uuid_index)

            # Retain null UUIDs and any UUID not listed for removal.
            keep_mask = pc.or_(
                pc.is_null(uuid_column),
                pc.invert(pc.is_in(uuid_column, value_set=bad_uuid_values)),
            )

            filtered_row_group = row_group.filter(keep_mask)
            removed_rows += row_group.num_rows - filtered_row_group.num_rows

            if filtered_row_group.num_rows:
                writer.write_table(filtered_row_group)

    # Confirm the rewritten fragment retained precisely the original schema.
    output_schema = pq.ParquetFile(destination_file).schema_arrow
    if not output_schema.equals(source_schema, check_metadata=True):
        raise RuntimeError(
            f"Schema changed while filtering {source_file.name}. "
            "Original data has not been modified."
        )

    return removed_rows


original_count, matching_rows = count_rows_and_matches(source_files)
print(f"Rows matching listed UUIDs: {matching_rows}")

if matching_rows == 0:
    print("No matching UUIDs found. Nothing changed.")

else:
    try:
        # Retain every non-Parquet file and all unaffected Parquet fragments.
        shutil.copytree(path, temp_path)

        rewritten_files = []
        removed_rows = 0

        for source_file in source_files:
            parquet = pq.ParquetFile(source_file)

            fragment_matches = 0
            for batch in parquet.iter_batches(columns=["uuid"]):
                fragment_matches += (
                    pc.sum(
                        pc.is_in(
                            batch.column(0),
                            value_set=bad_uuid_values,
                        )
                    ).as_py()
                    or 0
                )

            # Do not touch fragments which contain no UUIDs being removed.
            if fragment_matches == 0:
                continue

            relative_file = source_file.relative_to(path)
            replacement_file = temp_path / relative_file

            removed_rows += filter_parquet_file(
                source_file,
                replacement_file,
            )
            rewritten_files.append(relative_file)

        output_files = sorted(temp_path.rglob("*.parquet"))
        filtered_count, remaining_matches = count_rows_and_matches(output_files)

        if remaining_matches != 0:
            raise RuntimeError(
                f"Verification failed: {remaining_matches} matching rows remain."
            )

        if original_count - filtered_count != matching_rows:
            raise RuntimeError(
                "Verification failed: removed-row count does not match "
                "the number of matching source rows."
            )

        # Validate using the same dataset write operation used by RCS
        # consolidation. This occurs in a disposable directory; the original
        # dataset has not been changed at this point.
        part_scheme = ds.partitioning(
            schema=pa.schema([pa.field("date", pa.string())]),
            flavor="filename",
        )

        validation_dataset = ds.dataset(
            temp_path,
            format="parquet",
            partitioning=part_scheme,
        )

        ds.write_dataset(
            data=validation_dataset,
            base_dir=validation_path,
            format="parquet",
            partitioning=part_scheme,
            existing_data_behavior="overwrite_or_ignore",
        )

        # The validation write succeeded, so discard its disposable output.
        shutil.rmtree(validation_path)

        # Only now replace the original dataset and retain its backup.
        path.rename(backup_path)
        temp_path.rename(path)

        print(f"Removed rows:   {removed_rows}")
        print(f"Remaining rows: {filtered_count}")
        print(f"Rewritten fragments: {len(rewritten_files)}")

        for file_path in rewritten_files:
            print(f"  {file_path}")

        print(f"Updated data:   {path}")
        print(f"Backup:         {backup_path}")

    except Exception:
        # Leave the original untouched and remove incomplete temporary results.
        if temp_path.exists():
            shutil.rmtree(temp_path)

        if validation_path.exists():
            shutil.rmtree(validation_path)

        raise
